# AC14 Journey: Empirical Comparison Evidence

Canonical empirical-comparison journey notebook for AC14. This notebook is **artifact-backed**: it loads the real comparison contract, the real five-trial result, and the real trial artifacts instead of summarizing them in static dict literals.

## Journey Contract

- Journey id: `ac14_empirical_comparison_gate`
- Purpose: inspect the first real monolithic-vs-AC14 benchmark end to end
- Notebook mode: mixed planning/proof
- Canonical docs: `docs/plans/38_empirical_comparison_gate.md`, `docs/plans/43_full_trial_gate.md`, `docs/plans/44_verdict_interpretation_and_next_horizon.md`, `docs/plans/62_inconclusive_comparison_diagnosis.md`
- Main evidence root: `.ac14_out/full_trials_gate_1/`


In [1]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import json

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'docs' / 'plans' / 'CLAUDE.md').exists():
            return candidate
    raise RuntimeError('Could not locate the AC14 repo root from the current working directory.')

REPO_ROOT = find_repo_root(Path.cwd())
OUT_ROOT = REPO_ROOT / '.ac14_out' / 'full_trials_gate_1'
DECISION_PATH = OUT_ROOT / 'experiment_decision.json'
PLAN38_PATH = REPO_ROOT / 'docs' / 'plans' / '38_empirical_comparison_gate.md'
PLAN43_PATH = REPO_ROOT / 'docs' / 'plans' / '43_full_trial_gate.md'
PLAN44_PATH = REPO_ROOT / 'docs' / 'plans' / '44_verdict_interpretation_and_next_horizon.md'
PLAN62_PATH = REPO_ROOT / 'docs' / 'plans' / '62_inconclusive_comparison_diagnosis.md'
assert OUT_ROOT.exists(), f'Missing evidence root: {OUT_ROOT}'
assert DECISION_PATH.exists(), f'Missing decision artifact: {DECISION_PATH}'
assert PLAN38_PATH.exists() and PLAN43_PATH.exists() and PLAN44_PATH.exists() and PLAN62_PATH.exists()
decision_artifact = json.loads(DECISION_PATH.read_text())
decision_artifact['verdict']

'inconclusive'

## Phase 1: Comparison Contract And Verdict

- Purpose: load the frozen comparison contract and the final decision artifact
- Input -> output: frozen plan docs + `experiment_decision.json` -> one reviewable experiment summary
- Acceptance criteria: the decision artifact exists, the verdict is explicit, and the decision matches the committed control docs
- Status: `proven`
- Execution mode: `live`


In [2]:
assert decision_artifact['benchmark_id'] == 'order_exception_resolution_v1'
assert decision_artifact['trial_count'] == 5
assert decision_artifact['verdict'] in {'ac14_wins', 'monolithic_wins', 'inconclusive'}

experiment_summary = {
    'benchmark_id': decision_artifact['benchmark_id'],
    'trial_count': decision_artifact['trial_count'],
    'verdict': decision_artifact['verdict'],
    'rationale': decision_artifact['rationale'],
    'ac14_successes': decision_artifact['ac14']['successes'],
    'monolithic_successes': decision_artifact['monolithic']['successes'],
    'ac14_average_repair_loops': decision_artifact['ac14']['average_repair_loops'],
    'monolithic_average_repair_loops': decision_artifact['monolithic']['average_repair_loops'],
    'ac14_average_duration_s': decision_artifact['ac14']['average_duration_s'],
    'monolithic_average_duration_s': decision_artifact['monolithic']['average_duration_s'],
}
experiment_summary

{'benchmark_id': 'order_exception_resolution_v1',
 'trial_count': 5,
 'verdict': 'inconclusive',
 'rationale': 'The conditions tied on success and the secondary metrics did not separate them cleanly.',
 'ac14_successes': 2,
 'monolithic_successes': 2,
 'ac14_average_repair_loops': 1.6,
 'monolithic_average_repair_loops': 1.6,
 'ac14_average_duration_s': 111.3789599592099,
 'monolithic_average_duration_s': 84.69246088379877}

## Phase 2: Per-Trial Artifact Load

- Purpose: prove that the five paired trials are real and reviewable
- Input -> output: decision artifact -> paired trial reports -> one compact per-trial summary
- Acceptance criteria: every trial report path exists and the per-trial pass/fail story is explicit for both conditions
- Status: `proven`
- Execution mode: `live`


In [3]:
trial_rows = []
for report_path in decision_artifact['trial_report_paths']:
    absolute_path = REPO_ROOT / report_path
    assert absolute_path.exists(), f'Missing paired report: {absolute_path}'
    paired = json.loads(absolute_path.read_text())
    trial_rows.append({
        'trial_id': paired['trial_id'],
        'ac14_passed': paired['ac14']['passed'],
        'ac14_attempts': len(paired['ac14']['attempts']),
        'monolithic_passed': paired['monolithic']['passed'],
        'monolithic_attempts': len(paired['monolithic']['attempts']),
    })
trial_rows

[{'trial_id': 1,
  'ac14_passed': False,
  'ac14_attempts': 3,
  'monolithic_passed': False,
  'monolithic_attempts': 3},
 {'trial_id': 2,
  'ac14_passed': False,
  'ac14_attempts': 3,
  'monolithic_passed': False,
  'monolithic_attempts': 3},
 {'trial_id': 3,
  'ac14_passed': True,
  'ac14_attempts': 1,
  'monolithic_passed': False,
  'monolithic_attempts': 3},
 {'trial_id': 4,
  'ac14_passed': False,
  'ac14_attempts': 3,
  'monolithic_passed': True,
  'monolithic_attempts': 2},
 {'trial_id': 5,
  'ac14_passed': True,
  'ac14_attempts': 3,
  'monolithic_passed': True,
  'monolithic_attempts': 2}]

## Phase 3: Failure-Pattern Diagnosis

- Purpose: separate shared failure patterns from condition-specific ones
- Input -> output: attempt reports -> failure-pattern counters plus representative summaries
- Acceptance criteria: the notebook can name what both conditions kept failing on and what differed between them
- Status: `proven`
- Execution mode: `live`


In [4]:
def collect_failure_counts(side: str) -> Counter:
    counts: Counter = Counter()
    for report in OUT_ROOT.glob(f'trial_*/{side}/attempt_*/attempt_report.json'):
        data = json.loads(report.read_text())
        classification = data.get('failure_classification') or {}
        key = (classification.get('category'), classification.get('detail'))
        counts[key] += 1
    return counts

ac14_failure_counts = collect_failure_counts('ac14')
monolithic_failure_counts = collect_failure_counts('monolithic')

failure_diagnosis = {
    'ac14_top_failures': ac14_failure_counts.most_common(3),
    'monolithic_top_failures': monolithic_failure_counts.most_common(3),
    'verdict': decision_artifact['verdict'],
}
failure_diagnosis

{'ac14_top_failures': [(('packet_tests', 'packet-local tests failed'), 11),
  (('success', 'attempt passed the full benchmark harness'), 2)],
 'monolithic_top_failures': [(('packet_tests', 'packet-local tests failed'),
   9),
  (('success', 'attempt passed the full benchmark harness'), 2),
  (('generation',
    "monolithic generator did not return exactly one module per component: expected ['case_parser', 'customer_context_loader', 'exception_classifier', 'factor_correlator', 'inventory_risk_evaluator', 'manual_override_loader', 'priority_scorer', 'resolution_assembler', 'shipping_risk_evaluator'], got ['case_parser', 'exception_classifier', 'inventory_risk_evaluator', 'resolution_assembler']"),
   2)],
 'verdict': 'inconclusive'}

## Phase 4: Governance Lock

- Purpose: verify that the active docs reflect the real verdict and the current post-verdict lane
- Input -> output: active docs + verdict artifact -> one governance check
- Acceptance criteria: active docs point to Plan #63 as the current empirical contract, keep Plan #61 completed, and avoid drifting back to stale smoke-repair lanes
- Status: `proven`
- Execution mode: `live`


In [5]:
todo_text = (REPO_ROOT / 'docs' / 'TODO.md').read_text()
next24_text = (REPO_ROOT / 'docs' / 'AC14_NEXT_24_HOURS.md').read_text()
assert 'Plan #63: Runtime-First Comparison Contract' in todo_text
assert 'Plan #61: Executable Journey Notebook Remediation' in todo_text
assert 'Plan #63: Runtime-First Comparison Contract' in next24_text
assert 'Plan #61: Executable Journey Notebook Remediation' in next24_text

governance_check = {
    'active_empirical_contract': '63_runtime_first_comparison_contract',
    'completed_notebook_remediation': '61_executable_journey_notebook_remediation',
    'verdict': decision_artifact['verdict'],
}
governance_check


{'todo_active': 'Plan #61: Executable Journey Notebook Remediation',
 'todo_follow_on': 'Plan #63: Runtime-First Comparison Contract',
 'verdict': 'inconclusive'}

## Decision Summary

- The first empirical gate is complete and `inconclusive`.
- The notebook now runs through the real decision artifact and the real paired-trial artifacts.
- The notebook-remediation lane is complete, and the runtime-first comparison contract is the active next empirical lane.
